# 🚀 MASSIVE SOTA BENCHMARK: STT, VAD & VOICE EMBEDDINGS
### Hệ thống Benchmark tự động đo đạc và xếp hạng động (Dynamic Leaderboard)
Notebook này thực hiện:
1. **Benchmark hàng loạt 17+ mô hình STT** (PhoWhisper, Whisper, Distil-Whisper, CrisperWhisper, Wav2Vec2, Meta-MMS).
2. **Benchmark VAD & Diarization** (Silero-VAD với ONNX runtime, Pyannote).
3. **Benchmark Voice Embedding** (SpeechBrain: ECAPA-TDNN, ResNet, X-Vector).
4. **Mục 6 - Tổng kết động**: Tự động tính toán và xuất **TOP 5** theo từng tiêu chí (Tốc độ suy luận, Thời gian nạp, Chuyên biệt tiếng Việt, Đa ngữ) hoàn toàn bằng code, không fix cứng!


In [1]:
# 📦 Cài đặt đầy đủ các thư viện (đã bổ sung onnxruntime để Silero-VAD chạy trơn tru)
!pip install -q transformers torchaudio librosa speechbrain funasr onnxruntime pandas accelerate gTTS

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.8/298.8 kB 7.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 6.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 999.0/999.0 kB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5

## 1. Chuẩn bị dữ liệu mẫu tự động (Audio Samples)

In [2]:
import os
import IPython.display as ipd
from gtts import gTTS
import torchaudio
import torch

AUDIO_EN = "/kaggle/working/micro-machines.wav"
if not os.path.exists(AUDIO_EN):
    !wget -q -O {AUDIO_EN} https://cdn.openai.com/whisper/draft-20220913a/micro-machines.wav

AUDIO_VI = "/kaggle/working/vietnamese-sample.wav"
if not os.path.exists(AUDIO_VI):
    temp_mp3 = "/kaggle/working/temp_vi.mp3"
    tts_vi = gTTS(
        "Xin chào, đây là một bài kiểm tra toàn diện cho hệ thống nhận dạng giọng nói và tách người nói. "
        "Chúng ta sẽ cùng đánh giá tốc độ và độ chính xác của từng mô hình.",
        lang='vi'
    )
    tts_vi.save(temp_mp3)
    sig, sr = torchaudio.load(temp_mp3)
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
        sig = resampler(sig)
    torchaudio.save(AUDIO_VI, sig, 16000)
    if os.path.exists(temp_mp3):
        os.remove(temp_mp3)

print("✅ Sẵn sàng 2 mẫu âm thanh chuẩn 16kHz WAV:")
print("1. Tiếng Việt:", AUDIO_VI)
print("2. Tiếng Anh:", AUDIO_EN)


✅ Sẵn sàng 2 mẫu âm thanh chuẩn 16kHz WAV:
1. Tiếng Việt: /kaggle/working/vietnamese-sample.wav
2. Tiếng Anh: /kaggle/working/micro-machines.wav


## 2. STT Benchmark Hàng Loạt (Speech-to-Text)

In [3]:
import time
import gc
import warnings
import torch
import pandas as pd
from transformers import pipeline

warnings.filterwarnings("ignore")
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Sử dụng thiết bị phần cứng: {device}")

stt_models = [
    # --- Họ PhoWhisper (Tiếng Việt chuyên biệt) ---
    {"id": "vinai/PhoWhisper-tiny", "family": "PhoWhisper", "arch": "Seq2Seq", "target": "VI", "use_safetensors": False},
    {"id": "vinai/PhoWhisper-base", "family": "PhoWhisper", "arch": "Seq2Seq", "target": "VI", "use_safetensors": False},
    {"id": "vinai/PhoWhisper-small", "family": "PhoWhisper", "arch": "Seq2Seq", "target": "VI", "use_safetensors": False},
    {"id": "vinai/PhoWhisper-large", "family": "PhoWhisper", "arch": "Seq2Seq", "target": "VI", "use_safetensors": False},

    # --- Họ OpenAI Whisper chuẩn ---
    {"id": "openai/whisper-tiny", "family": "Whisper", "arch": "Seq2Seq", "target": "Multi", "use_safetensors": True},
    {"id": "openai/whisper-base", "family": "Whisper", "arch": "Seq2Seq", "target": "Multi", "use_safetensors": True},
    {"id": "openai/whisper-small", "family": "Whisper", "arch": "Seq2Seq", "target": "Multi", "use_safetensors": True},
    {"id": "openai/whisper-medium", "family": "Whisper", "arch": "Seq2Seq", "target": "Multi", "use_safetensors": True},
    {"id": "openai/whisper-large-v3-turbo", "family": "Whisper-Turbo", "arch": "Seq2Seq", "target": "Multi", "use_safetensors": True},
    {"id": "openai/whisper-large-v3", "family": "Whisper", "arch": "Seq2Seq", "target": "Multi", "use_safetensors": True},

    # --- Họ Distil-Whisper (Nén tốc độ cao) ---
    {"id": "distil-whisper/distil-small.en", "family": "Distil-Whisper", "arch": "Seq2Seq", "target": "EN", "use_safetensors": True},
    {"id": "distil-whisper/distil-medium.en", "family": "Distil-Whisper", "arch": "Seq2Seq", "target": "EN", "use_safetensors": True},
    {"id": "distil-whisper/distil-large-v3", "family": "Distil-Whisper", "arch": "Seq2Seq", "target": "EN", "use_safetensors": True},

    # --- Họ CrisperWhisper ---
    {"id": "nyralabs/CrisperWhisper2.0_large", "family": "CrisperWhisper", "arch": "Seq2Seq", "target": "Multi", "use_safetensors": True},

    # --- Họ Wav2Vec2 & MMS (Kiến trúc CTC siêu nhanh) ---
    {"id": "nguyenvulebinh/wav2vec2-base-vietnamese-250h", "family": "Wav2Vec2", "arch": "CTC", "target": "VI", "use_safetensors": False},
    {"id": "facebook/wav2vec2-base-960h", "family": "Wav2Vec2", "arch": "CTC", "target": "EN", "use_safetensors": False},
    {"id": "facebook/mms-1b-all", "family": "Meta-MMS", "arch": "CTC", "target": "Multi", "use_safetensors": True}
]

stt_results = []
TEST_AUDIO = AUDIO_VI

print(f"Bắt đầu chạy benchmark hàng loạt {len(stt_models)} mô hình STT...")

for idx, m in enumerate(stt_models, 1):
    model_id = m["id"]
    print(f"\n[{idx}/{len(stt_models)}] Đang test: {model_id} ({m['family']})...")
    
    status = "Thành công"
    load_time = None
    inf_time = None
    text_output = ""
    
    try:
        t0 = time.time()
        pipe_kwargs = {"use_safetensors": m["use_safetensors"]} if m["use_safetensors"] is not None else {}
        pipe = pipeline(
            "automatic-speech-recognition",
            model=model_id,
            device=device,
            chunk_length_s=30,
            model_kwargs=pipe_kwargs
        )
        load_time = round(time.time() - t0, 2)
        
        t1 = time.time()
        res = pipe(TEST_AUDIO)
        inf_time = round(time.time() - t1, 2)
        
        text_output = res.get("text", "").strip()
        print(f"   -> Load: {load_time}s | Infer: {inf_time}s")
        print(f"   -> Kết quả: {text_output[:80]}...")
        
    except Exception as e:
        status = "Lỗi"
        text_output = f"Error: {str(e)[:120]}"
        print(f"   -> ⚠️ Lỗi khi chạy {model_id}: {str(e)[:80]}")
        
    finally:
        stt_results.append({
            "STT Model": model_id,
            "Family": m["family"],
            "Arch": m["arch"],
            "Target Lang": m["target"],
            "Status": status,
            "Load Time (s)": load_time,
            "Inference Time (s)": inf_time,
            "Transcription": text_output
        })
        if 'pipe' in locals():
            del pipe
        torch.cuda.empty_cache()
        gc.collect()

print("\n🏆 HOÀN TẤT BENCHMARK STT!")
stt_leaderboard = pd.DataFrame(stt_results)
display(stt_leaderboard)


Sử dụng thiết bị phần cứng: cuda:0
Bắt đầu chạy benchmark hàng loạt 17 mô hình STT...

[1/17] Đang test: vinai/PhoWhisper-tiny (PhoWhisper)...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/151M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/168 [00:00<?, ?it/s]

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 761, in hf_raise_for_status
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '403 Forbidden' for url 'https://huggingface.co/api/models/vinai/PhoWhisper-tiny/discussions?p=0'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py

...

vocab.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

   -> Load: 26.94s | Infer: 0.8s
   -> Kết quả: sin chao ay la mt bai kicm cra toan gian cho hi thung nhan gian giong noi va tac...

🏆 HOÀN TẤT BENCHMARK STT!


,STT Model,Family,Arch,Target Lang,Status,Load Time (s),Inference Time (s),Transcription
0,vinai/PhoWhisper-tiny,PhoWhisper,Seq2Seq,VI,Thành công,5.09,2.57,xin chào đây là một bài kiểm tra toàn diện cho...
1,vinai/PhoWhisper-base,PhoWhisper,Seq2Seq,VI,Thành công,4.76,0.95,xin chào đây là một bài kiểm tra toàn diện cho...
2,vinai/PhoWhisper-small,PhoWhisper,Seq2Seq,VI,Thành công,7.17,1.90,xin chào đây là một bài kiểm tra toàn diện cho...
3,vinai/PhoWhisper-large,PhoWhisper,Seq2Seq,VI,Thành công,26.67,7.89,xin chào đây là một bài kiểm tra toàn diện cho...
4,openai/whisper-tiny,Whisper,Seq2Seq,Multi,Thành công,3.36,0.86,"Xin chào, đây là một bay kiểm tra toàn diện ch..."
5,openai/whisper-base,Whisper,Seq2Seq,Multi,Thành công,4.20,0.91,"Xin chào, đây là một bài kiểm tra toàn diện ch..."
6,openai/whisper-small,Whisper,Seq2Seq,Multi,Thành công,7.44,1.88,"Xin chào, đây là một bài kiểm tra toàn diện ch..."
7,openai/whisper-medium,Whisper,Seq2Seq,Multi,Thành công,13.55,4.10,"Xin chào, đây là một bài kiểm tra toàn diện ch..."
8,openai/whisper-large-v3-turbo,Whisper-Turbo,Seq2Seq,Multi,Thành công,11.74,1.67,"Xin chào, đây là một bài kiểm tra toàn diện ch..."
9,openai/whisper-large-v3,Whisper,Seq2Seq,Multi,Thành công,20.74,3.56,"Xin chào, đây là một bài kiểm tra toàn diện ch..."


## 3. STT Đặc Thù: SenseVoiceSmall (Alibaba FunAudioLLM qua Hugging Face Hub)

In [4]:
try:
    from funasr import AutoModel
    print("Đang tải FunAudioLLM/SenseVoiceSmall từ Hugging Face Hub (hub='hf')...")
    t0 = time.time()
    # Chỉ định rõ hub='hf' để tránh lỗi ModelScope ở nước ngoài
    model_sense = AutoModel(model="FunAudioLLM/SenseVoiceSmall", hub="hf", trust_remote_code=True, device=device)
    load_t = round(time.time() - t0, 2)
    
    t1 = time.time()
    res_sense = model_sense.generate(input=AUDIO_VI, batch_size_s=300)
    inf_t = round(time.time() - t1, 2)
    
    print(f"✅ SenseVoiceSmall hoàn tất: Load {load_t}s, Infer {inf_t}s")
    print("Văn bản nhận diện:", res_sense)
    
    # Bổ sung kết quả vào leaderboard STT nếu thành công
    stt_leaderboard = pd.concat([stt_leaderboard, pd.DataFrame([{
        "STT Model": "FunAudioLLM/SenseVoiceSmall",
        "Family": "SenseVoice",
        "Arch": "Non-Autoregressive",
        "Target Lang": "Multi",
        "Status": "Thành công",
        "Load Time (s)": load_t,
        "Inference Time (s)": inf_t,
        "Transcription": str(res_sense)[:120]
    }])], ignore_index=True)
    
    del model_sense
    torch.cuda.empty_cache()
    gc.collect()
except Exception as e:
    print("⚠️ SenseVoice test skipped / error:", e)


2026-09-15 10:04:11,717 [INFO] download models from model hub: hf
2026-09-15 10:04:11,802 [INFO] HTTP Request: GET https://huggingface.co/api/models/FunAudioLLM/SenseVoiceSmall/revision/main "HTTP/1.1 200 OK"


Đang tải FunAudioLLM/SenseVoiceSmall từ Hugging Face Hub (hub='hf')...
funasr version: 1.4.15.
Check update of funasr, and it would cost few times. You may disable it by set `disable_update=True` in AutoModel
You are using the latest version of funasr-1.4.15


Fetching 29 files:   0%|          | 0/29 [00:00<?, ?it/s]

2026-09-15 10:04:11,855 [INFO] HTTP Request: HEAD https://huggingface.co/FunAudioLLM/SenseVoiceSmall/resolve/3847d57b6bdf2dd8875cb1508d2af43d80a16bf7/.gitattributes "HTTP/1.1 307 Temporary Redirect"
2026-09-15 10:04:11,873 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/FunAudioLLM/SenseVoiceSmall/3847d57b6bdf2dd8875cb1508d2af43d80a16bf7/.gitattributes "HTTP/1.1 200 OK"
2026-09-15 10:04:11,891 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/FunAudioLLM/SenseVoiceSmall/3847d57b6bdf2dd8875cb1508d2af43d80a16bf7/.gitattributes "HTTP/1.1 200 OK"
2026-09-15 10:04:11,900 [INFO] HTTP Request: HEAD https://huggingface.co/FunAudioLLM/SenseVoiceSmall/resolve/3847d57b6bdf2dd8875cb1508d2af43d80a16bf7/README.md "HTTP/1.1 307 Temporary Redirect"
2026-09-15 10:04:11,903 [INFO] HTTP Request: HEAD https://huggingface.co/FunAudioLLM/SenseVoiceSmall/resolve/3847d57b6bdf2dd8875cb1508d2af43d80a16bf7/am.mvn "HTTP/1.1 307 Temporary Redirect"
2026-09-15 10:04:

Detect model requirements, begin to install it: /root/.cache/huggingface/hub/models--FunAudioLLM--SenseVoiceSmall/snapshots/3847d57b6bdf2dd8875cb1508d2af43d80a16bf7/requirements.txt
install model requirements successfully


2026-09-15 10:04:31,582 [INFO] Loading pretrained params from /root/.cache/huggingface/hub/models--FunAudioLLM--SenseVoiceSmall/snapshots/3847d57b6bdf2dd8875cb1508d2af43d80a16bf7/model.pt
2026-09-15 10:04:31,589 [INFO] ckpt: /root/.cache/huggingface/hub/models--FunAudioLLM--SenseVoiceSmall/snapshots/3847d57b6bdf2dd8875cb1508d2af43d80a16bf7/model.pt
2026-09-15 10:04:32,761 [INFO] scope_map: ['module.', 'None']
2026-09-15 10:04:32,761 [INFO] excludes: None
2026-09-15 10:04:32,879 [INFO] Loading ckpt: /root/.cache/huggingface/hub/models--FunAudioLLM--SenseVoiceSmall/snapshots/3847d57b6bdf2dd8875cb1508d2af43d80a16bf7/model.pt, status: <All keys matched successfully>

100%|██████████| 1/1 [00:00<00:00,  3.97it/s]
{'load_data': '0.017', 'extract_feat': '0.031', 'forward': '0.252', 'batch_size': '1', 'rtf': '0.021'}, : 100%|██████████| 1/1 [00:00<00:00,  3.97it/s]
rtf_avg: 0.021: 100%|██████████| 1/1 [00:00<00:00,  3.82it/s]


✅ SenseVoiceSmall hoàn tất: Load 21.61s, Infer 0.27s
Văn bản nhận diện: [{'key': 'vietnamese-sample', 'text': '<|ko|><|NEUTRAL|><|Speech|><|woitn|>센 레라 못발 김차 또한 초희통人轴内发 의内 중타 색 공당사떡로 바로로 증고등冒행'}]


## 4. Voice Activity Detection (VAD) & Segmentation (Đã fix ONNX Runtime)

In [5]:
vad_results = []

# 1. Silero VAD (Được hỗ trợ đầy đủ qua onnxruntime / PyTorch JIT)
print("\n--- 1. Testing Silero-VAD ---")
try:
    t0 = time.time()
    # Tải model silero_vad chuẩn từ torch hub
    model_silero, utils = torch.hub.load(
        repo_or_dir='snakers4/silero-vad',
        model='silero_vad',
        force_reload=False,
        onnx=True  # onnxruntime đã được cài đặt sẵn ở Step 0
    )
    (get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = utils
    load_t = round(time.time() - t0, 2)
    
    t1 = time.time()
    wav = read_audio(AUDIO_VI, sampling_rate=16000)
    speech_timestamps = get_speech_timestamps(wav, model_silero, sampling_rate=16000)
    inf_t = round(time.time() - t1, 2)
    
    vad_results.append({
        "Model": "snakers4/silero-vad (ONNX)",
        "Task": "Voice Activity Detection (VAD)",
        "Load Time (s)": load_t,
        "Inference Time (s)": inf_t,
        "Result Preview": f"Phát hiện {len(speech_timestamps)} khoảng giọng nói: {speech_timestamps[:2]}"
    })
    print(f"✅ Silero-VAD chạy thành công! Tìm thấy {len(speech_timestamps)} đoạn thoại trong {inf_t}s")
    del model_silero
    torch.cuda.empty_cache()
    gc.collect()
except Exception as e:
    print("⚠️ Thử fallback Silero-VAD sang JIT không dùng ONNX...", e)
    try:
        t0 = time.time()
        model_silero, utils = torch.hub.load('snakers4/silero-vad', 'silero_vad', onnx=False)
        (get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = utils
        load_t = round(time.time() - t0, 2)
        t1 = time.time()
        wav = read_audio(AUDIO_VI, sampling_rate=16000)
        speech_timestamps = get_speech_timestamps(wav, model_silero, sampling_rate=16000)
        inf_t = round(time.time() - t1, 2)
        vad_results.append({
            "Model": "snakers4/silero-vad (PyTorch JIT)",
            "Task": "Voice Activity Detection (VAD)",
            "Load Time (s)": load_t,
            "Inference Time (s)": inf_t,
            "Result Preview": f"Phát hiện {len(speech_timestamps)} khoảng giọng nói: {speech_timestamps[:2]}"
        })
        print(f"✅ Silero-VAD (PyTorch JIT) chạy thành công! {len(speech_timestamps)} đoạn trong {inf_t}s")
        del model_silero
        torch.cuda.empty_cache()
        gc.collect()
    except Exception as e2:
        print("❌ Lỗi cả 2 phương thức Silero-VAD:", e2)
        vad_results.append({"Model": "snakers4/silero-vad", "Task": "VAD", "Load Time (s)": None, "Inference Time (s)": None, "Result Preview": str(e2)[:100]})

# 2. Pyannote Segmentation / Diarization
print("\n--- 2. Testing Pyannote Diarization ---")
try:
    from pyannote.audio import Pipeline
    hf_token = os.environ.get("HF_TOKEN", None)
    if hf_token:
        t0 = time.time()
        pipeline_pyannote = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", use_auth_token=hf_token)
        if torch.cuda.is_available():
            pipeline_pyannote.to(torch.device("cuda"))
        load_t = round(time.time() - t0, 2)
        
        t1 = time.time()
        diar_out = pipeline_pyannote(AUDIO_VI)
        inf_t = round(time.time() - t1, 2)
        
        segments = []
        for turn, _, spk in list(diar_out.itertracks(yield_label=True))[:3]:
            segments.append(f"[{turn.start:.1f}s - {turn.end:.1f}s]: {spk}")
            
        vad_results.append({
            "Model": "pyannote/speaker-diarization-3.1",
            "Task": "Speaker Diarization",
            "Load Time (s)": load_t,
            "Inference Time (s)": inf_t,
            "Result Preview": " | ".join(segments)
        })
        del pipeline_pyannote
        torch.cuda.empty_cache()
        gc.collect()
    else:
        vad_results.append({
            "Model": "pyannote/speaker-diarization-3.1",
            "Task": "Speaker Diarization",
            "Load Time (s)": None,
            "Inference Time (s)": None,
            "Result Preview": "Bỏ qua (Chưa cấu hình HF_TOKEN)"
        })
        print("Bỏ qua Pyannote vì chưa có HF_TOKEN")
except Exception as e:
    vad_results.append({"Model": "pyannote/speaker-diarization-3.1", "Task": "Diarization", "Load Time (s)": None, "Inference Time (s)": None, "Result Preview": str(e)[:100]})

vad_df = pd.DataFrame(vad_results)
display(vad_df)



--- 1. Testing Silero-VAD ---
Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip
✅ Silero-VAD chạy thành công! Tìm thấy 3 đoạn thoại trong 0.17s

--- 2. Testing Pyannote Diarization ---


,Model,Task,Load Time (s),Inference Time (s),Result Preview
0,snakers4/silero-vad (ONNX),Voice Activity Detection (VAD),1.51,0.17,"Phát hiện 3 khoảng giọng nói: [{'start': 2592,..."
1,pyannote/speaker-diarization-3.1,Diarization,NaN,NaN,No module named 'pyannote'


## 5. Voice Embedding & Speaker Verification (SpeechBrain)

In [6]:
from speechbrain.inference.speaker import EncoderClassifier

embedding_models = [
    {"id": "speechbrain/spkrec-ecapa-voxceleb", "name": "ECAPA-TDNN (192-dim)", "desc": "Chuẩn công nghiệp, nhẹ, tốc độ cao"},
    {"id": "speechbrain/spkrec-resnet-voxceleb", "name": "ResNet (256-dim)", "desc": "Trích xuất đặc trưng sâu, kháng nhiễu"},
    {"id": "speechbrain/spkrec-xvect-voxceleb", "name": "X-Vector (512-dim)", "desc": "Kiến trúc cổ điển kinh điển"}
]

embed_results = []
signal_en, fs = torchaudio.load(AUDIO_EN)

for m in embedding_models:
    model_id = m["id"]
    print(f"\nĐang test Voice Embedding: {model_id}...")
    try:
        t0 = time.time()
        classifier = EncoderClassifier.from_hparams(source=model_id, run_opts={"device": device})
        load_t = round(time.time() - t0, 2)
        
        t1 = time.time()
        emb = classifier.encode_batch(signal_en)
        inf_t = round(time.time() - t1, 2)
        
        embed_results.append({
            "Model": model_id,
            "Architecture": m["name"],
            "Description": m["desc"],
            "Load Time (s)": load_t,
            "Inference Time (s)": inf_t,
            "Embedding Vector Shape": str(list(emb.shape))
        })
        print(f"   -> Xong! Shape: {list(emb.shape)}, Time: {inf_t}s")
        
        del classifier
        torch.cuda.empty_cache()
        gc.collect()
    except Exception as e:
        print(f"   -> ⚠️ Lỗi {model_id}: {e}")
        embed_results.append({
            "Model": model_id,
            "Architecture": m["name"],
            "Description": m["desc"],
            "Load Time (s)": None,
            "Inference Time (s)": None,
            "Embedding Vector Shape": str(e)[:60]
        })

embed_leaderboard = pd.DataFrame(embed_results)
display(embed_leaderboard)


2026-09-15 10:04:36,532 [INFO] Applied quirks (see `speechbrain.utils.quirks`): [disable_jit_profiling, allow_tf32]
2026-09-15 10:04:36,533 [INFO] Excluded quirks specified by the `SB_DISABLE_QUIRKS` environment (comma-separated list): []
2026-09-15 10:04:36,625 [INFO] Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
2026-09-15 10:04:36,707 [INFO] HTTP Request: HEAD https://huggingface.co/speechbrain/spkrec-ecapa-voxceleb/resolve/main/hyperparams.yaml "HTTP/1.1 307 Temporary Redirect"
2026-09-15 10:04:36,736 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/speechbrain/spkrec-ecapa-voxceleb/0f99f2d0ebe89ac095bcc5903c4dd8f72b367286/hyperparams.yaml "HTTP/1.1 200 OK"
2026-09-15 10:04:36,754 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/speechbrain/spkrec-ecapa-voxceleb/0f99f2d0ebe89ac095bcc5903c4dd8f72b367286/hyperparams.yaml "HTTP/1.1 200 OK"



Đang test Voice Embedding: speechbrain/spkrec-ecapa-voxceleb...


hyperparams.yaml: 0.00B [00:00, ?B/s]

2026-09-15 10:04:36,807 [INFO] HTTP Request: HEAD https://huggingface.co/speechbrain/spkrec-ecapa-voxceleb/resolve/main/hyperparams.yaml "HTTP/1.1 307 Temporary Redirect"
2026-09-15 10:04:36,823 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/speechbrain/spkrec-ecapa-voxceleb/0f99f2d0ebe89ac095bcc5903c4dd8f72b367286/hyperparams.yaml "HTTP/1.1 200 OK"
2026-09-15 10:04:37,004 [INFO] Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
2026-09-15 10:04:37,045 [INFO] HTTP Request: HEAD https://huggingface.co/speechbrain/spkrec-ecapa-voxceleb/resolve/main/embedding_model.ckpt "HTTP/1.1 302 Found"
2026-09-15 10:04:37,083 [INFO] HTTP Request: GET https://huggingface.co/api/models/speechbrain/spkrec-ecapa-voxceleb/xet-read-token/0f99f2d0ebe89ac095bcc5903c4dd8f72b367286 "HTTP/1.1 200 OK"


embedding_model.ckpt:   0%|          | 0.00/83.3M [00:00<?, ?B/s]

...

2026-09-15 10:04:45,702 [INFO] HTTP Request: HEAD https://huggingface.co/speechbrain/spkrec-xvect-voxceleb/resolve/main/classifier.ckpt "HTTP/1.1 302 Found"
2026-09-15 10:04:45,705 [INFO] Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
2026-09-15 10:04:45,750 [INFO] HTTP Request: HEAD https://huggingface.co/speechbrain/spkrec-xvect-voxceleb/resolve/main/label_encoder.txt "HTTP/1.1 307 Temporary Redirect"
2026-09-15 10:04:45,767 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/speechbrain/spkrec-xvect-voxceleb/56895a2df401be4150a159f3a1c653f00051d477/label_encoder.txt "HTTP/1.1 200 OK"
2026-09-15 10:04:45,787 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/speechbrain/spkrec-xvect-voxceleb/56895a2df401be4150a159f3a1c653f00051d477/label_encoder.txt "HTTP/1.1 200 OK"


label_encoder.txt: 0.00B [00:00, ?B/s]

2026-09-15 10:04:45,841 [INFO] HTTP Request: HEAD https://huggingface.co/speechbrain/spkrec-xvect-voxceleb/resolve/main/label_encoder.txt "HTTP/1.1 307 Temporary Redirect"
2026-09-15 10:04:45,858 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/speechbrain/spkrec-xvect-voxceleb/56895a2df401be4150a159f3a1c653f00051d477/label_encoder.txt "HTTP/1.1 200 OK"
2026-09-15 10:04:45,860 [INFO] Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


   -> Xong! Shape: [2, 1, 512], Time: 0.08s


,Model,Architecture,Description,Load Time (s),Inference Time (s),Embedding Vector Shape
0,speechbrain/spkrec-ecapa-voxceleb,ECAPA-TDNN (192-dim),"Chuẩn công nghiệp, nhẹ, tốc độ cao",2.06,0.60,"[2, 1, 192]"
1,speechbrain/spkrec-resnet-voxceleb,ResNet (256-dim),"Trích xuất đặc trưng sâu, kháng nhiễu",2.16,1.28,"[2, 256]"
2,speechbrain/spkrec-xvect-voxceleb,X-Vector (512-dim),Kiến trúc cổ điển kinh điển,2.28,0.08,"[2, 1, 512]"


## 6. Tổng Kết Động: Top 5 Bảng Xếp Hạng Theo Từng Tiêu Chí
Phần này **tính toán tự động 100% từ kết quả chạy thực tế ở trên**, không fix cứng bất kỳ giá trị nào.

In [7]:
print("="*80)
print("🥇 BẢNG XẾP HẠNG TỔNG KẾT TỰ ĐỘNG (DYNAMIC LEADERBOARD)")
print("="*80)

# Lọc các model chạy thành công
stt_success = stt_leaderboard[stt_leaderboard["Status"] == "Thành công"].copy()

# -------------------------------------------------------------
# 1. TOP 5 STT SUY LUẬN NHANH NHẤT (Inference Time)
# -------------------------------------------------------------
print("\n⚡ [STT] TOP 5 MÔ HÌNH SUY LUẬN NHANH NHẤT (Inference Time):")
top5_stt_speed = stt_success.sort_values(by="Inference Time (s)").head(5)[
    ["STT Model", "Family", "Arch", "Inference Time (s)", "Transcription"]
]
display(top5_stt_speed)

# -------------------------------------------------------------
# 2. TOP 5 STT NẠP MODEL NHANH NHẤT (Load Time)
# -------------------------------------------------------------
print("\n🚀 [STT] TOP 5 MÔ HÌNH NẠP BỘ NHỚ NHANH NHẤT (Load Time):")
top5_stt_load = stt_success.sort_values(by="Load Time (s)").head(5)[
    ["STT Model", "Family", "Arch", "Load Time (s)", "Inference Time (s)"]
]
display(top5_stt_load)

# -------------------------------------------------------------
# 3. TOP 5 STT TIẾNG VIỆT (Target Lang = VI hoặc Multi) TỐC ĐỘ CAO NHẤT
# -------------------------------------------------------------
print("\n🇻🇳 [STT] TOP 5 MÔ HÌNH HỖ TRỢ TIẾNG VIỆT TỐT NHẤT:")
vn_models = stt_success[stt_success["Target Lang"].isin(["VI", "Multi"])].copy()
top5_vn = vn_models.sort_values(by="Inference Time (s)").head(5)[
    ["STT Model", "Family", "Target Lang", "Inference Time (s)", "Transcription"]
]
display(top5_vn)

# -------------------------------------------------------------
# 4. TOP 5 VOICE EMBEDDING / SPEAKER RECOGNITION (Tốc độ trích xuất vector)
# -------------------------------------------------------------
print("\n🎯 [VOICE EMBEDDING] BẢNG XẾP HẠNG MÔ HÌNH TRÍCH XUẤT ĐẶC TRƯNG GIỌNG NÓI:")
embed_success = embed_leaderboard[embed_leaderboard["Inference Time (s)"].notna()].copy()
if not embed_success.empty:
    top_embed = embed_success.sort_values(by="Inference Time (s)").head(5)[
        ["Model", "Architecture", "Inference Time (s)", "Embedding Vector Shape", "Description"]
    ]
    display(top_embed)
else:
    print("Chưa có kết quả voice embedding thành công.")

# -------------------------------------------------------------
# 5. KẾT QUẢ VAD / DIARIZATION
# -------------------------------------------------------------
print("\n🎙️ [VAD & DIARIZATION] BẢNG XẾP HẠNG PHÁT HIỆN TIẾNG NÓI & PHÂN TÁCH NGƯỜI NÓI:")
vad_success = vad_df[vad_df["Inference Time (s)"].notna()].copy()
if not vad_success.empty:
    top_vad = vad_success.sort_values(by="Inference Time (s)").head(5)[
        ["Model", "Task", "Inference Time (s)", "Result Preview"]
    ]
    display(top_vad)
else:
    display(vad_df)


🥇 BẢNG XẾP HẠNG TỔNG KẾT TỰ ĐỘNG (DYNAMIC LEADERBOARD)

⚡ [STT] TOP 5 MÔ HÌNH SUY LUẬN NHANH NHẤT (Inference Time):


,STT Model,Family,Arch,Inference Time (s),Transcription
15,facebook/wav2vec2-base-960h,Wav2Vec2,CTC,0.22,SINCHAU LE LAMUD BAY KIMCHA PANSIEN CHOHITOM Y...
17,FunAudioLLM/SenseVoiceSmall,SenseVoice,Non-Autoregressive,0.27,"[{'key': 'vietnamese-sample', 'text': '<|ko|><..."
14,nguyenvulebinh/wav2vec2-base-vietnamese-250h,Wav2Vec2,CTC,0.29,xin chào đây là một bài kiểm tra toàn diện cho...
12,distil-whisper/distil-large-v3,Distil-Whisper,Seq2Seq,0.72,"SIN CHAL, this is a one of BAYMTHA TOANZAN TOE..."
10,distil-whisper/distil-small.en,Distil-Whisper,Seq2Seq,0.73,"Sin Chau, Dailamudai Kim Cha, Tuan Zian Chojek..."



🚀 [STT] TOP 5 MÔ HÌNH NẠP BỘ NHỚ NHANH NHẤT (Load Time):


,STT Model,Family,Arch,Load Time (s),Inference Time (s)
4,openai/whisper-tiny,Whisper,Seq2Seq,3.36,0.86
5,openai/whisper-base,Whisper,Seq2Seq,4.20,0.91
1,vinai/PhoWhisper-base,PhoWhisper,Seq2Seq,4.76,0.95
0,vinai/PhoWhisper-tiny,PhoWhisper,Seq2Seq,5.09,2.57
14,nguyenvulebinh/wav2vec2-base-vietnamese-250h,Wav2Vec2,CTC,5.41,0.29



🇻🇳 [STT] TOP 5 MÔ HÌNH HỖ TRỢ TIẾNG VIỆT TỐT NHẤT:


,STT Model,Family,Target Lang,Inference Time (s),Transcription
17,FunAudioLLM/SenseVoiceSmall,SenseVoice,Multi,0.27,"[{'key': 'vietnamese-sample', 'text': '<|ko|><..."
14,nguyenvulebinh/wav2vec2-base-vietnamese-250h,Wav2Vec2,VI,0.29,xin chào đây là một bài kiểm tra toàn diện cho...
16,facebook/mms-1b-all,Meta-MMS,Multi,0.80,sin chao ay la mt bai kicm cra toan gian cho h...
4,openai/whisper-tiny,Whisper,Multi,0.86,"Xin chào, đây là một bay kiểm tra toàn diện ch..."
5,openai/whisper-base,Whisper,Multi,0.91,"Xin chào, đây là một bài kiểm tra toàn diện ch..."



🎯 [VOICE EMBEDDING] BẢNG XẾP HẠNG MÔ HÌNH TRÍCH XUẤT ĐẶC TRƯNG GIỌNG NÓI:


,Model,Architecture,Inference Time (s),Embedding Vector Shape,Description
2,speechbrain/spkrec-xvect-voxceleb,X-Vector (512-dim),0.08,"[2, 1, 512]",Kiến trúc cổ điển kinh điển
0,speechbrain/spkrec-ecapa-voxceleb,ECAPA-TDNN (192-dim),0.60,"[2, 1, 192]","Chuẩn công nghiệp, nhẹ, tốc độ cao"
1,speechbrain/spkrec-resnet-voxceleb,ResNet (256-dim),1.28,"[2, 256]","Trích xuất đặc trưng sâu, kháng nhiễu"



🎙️ [VAD & DIARIZATION] BẢNG XẾP HẠNG PHÁT HIỆN TIẾNG NÓI & PHÂN TÁCH NGƯỜI NÓI:


,Model,Task,Inference Time (s),Result Preview
0,snakers4/silero-vad (ONNX),Voice Activity Detection (VAD),0.17,"Phát hiện 3 khoảng giọng nói: [{'start': 2592,..."
